In [0]:
from pyspark.sql.functions import col,sum,avg,floor,max,row_number,rank,dense_rank,explode,count
from pyspark.sql.window import Window
df_employee=spark.table("employee")
df_dept=spark.table("department")
df_customer=spark.table("customer")
df_order=spark.table("orders")   
df_invoice=spark.table("invoice")


In [0]:
#Get employee name with department name
df_employee.alias("e").join(df_dept.alias("d"),col("e.dept_id")==col("d.dept_id"),"inner").select("e.emp_name","d.dept_name").show()

In [0]:
#Get all employees including those without departments
df_employee.alias("e").join(df_dept.alias("d"), col("e.dept_id")==col("d.dept_id"),"left").select("e.emp_name","e.dept_id","d.dept_name").show()



In [0]:
# Find total salary department-wise
df_employee.groupBy("dept_id").agg(sum("salary")).show()

In [0]:
#Find average salary by department
df_employee.groupBy("dept_id").agg(floor(avg("salary"))).show()

In [0]:
#Find highest salary in each department
df_employee.groupBy("dept_id").agg(max("salary")).orderBy(col("dept_id").desc()).show()



In [0]:
#Rank employees by salary within department
from pyspark.sql.window import Window
from pyspark.sql.functions import dense_rank
df_employee.withColumn("rank",dense_rank().over(Window.partitionBy("dept_id").orderBy(col("salary").desc()))).select("emp_name","dept_id","salary","rank").show()


In [0]:
#Find highest paid employee in each department
df_employee.withColumn("rank",dense_rank().over(Window.partitionBy("dept_id").orderBy(col("salary").desc()))).filter(col("rank")==1).select("emp_name","dept_id","salary").show()

In [0]:
#Running total of salaries department-wise
df_employee.withColumn("running",sum(col("salary")).over(Window.partitionBy("dept_id").orderBy(col("salary").desc()))).select("emp_name","dept_id","salary","running").show()

In [0]:
df_customer=spark.table("customer")
df_order=spark.table("orders")   
df_invoice=spark.table("invoice")

In [0]:
#Get customer name with order details
from pyspark.sql.functions import col,row_number,dense_rank,rank,avg,floor,max,sum
from pyspark.sql.window import Window

df_customer.alias("c").join(df_order.alias("o"),col("c.cust_id")==col("o.cust_id"),"inner").select("c.cust_name","o.order_id","o.order_date","o.order_amount").show()


In [0]:
#Get customer orders with delivery date
df_customer.alias("c").join(df_order.alias("o"),col("c.cust_id")==col("o.cust_id"),"inner").join(df_invoice.alias("i"),col("o.order_id")==col("i.order_id"),"inner").select("c.cust_name","o.order_id","o.order_date","o.order_amount","i.delivery_date").display()



In [0]:
#Find customers who never placed orders
df_customer.alias("c").join(df_order.alias("o"),col("c.cust_id")==col("o.cust_id"),"outer").filter(col("o.order_id").isNull()).select("c.cust_name").display()

df_customer.alias("c").join(df_order.alias("o"),col("c.cust_id")==col("o.cust_id"),"left_anti").select("c.cust_name").display()


In [0]:
#Find orders not yet delivered
df_customer.alias("c").join(df_order.alias("o"),col("c.cust_id")==col("o.cust_id"),"inner").join(df_invoice.alias("i"),col("o.order_id")==col("i.order_id"),"inner").filter(col("i.delivery_date").isNull()).select("c.cust_name","o.order_id","o.order_date","o.order_amount","i.delivery_date").display()

In [0]:
#Find total order amount by customer
df_customer.alias("c").join(df_order.alias("o"),col("c.cust_id")==col("o.cust_id"),"inner").groupBy("c.cust_id","c.cust_name").agg(sum(col("o.order_amount")).alias("total_order_amount")).orderBy(col("c.cust_id").asc()).select("c.cust_id","c.cust_name","total_order_amount").display()

df_customer.alias("c").join(df_order.alias("o"),col("c.cust_id")==col("o.cust_id"),"inner") \
    .withColumn("total_order_amount",sum(col("o.order_amount")).over(Window.partitionBy("c.cust_id"))).select("c.cust_id","c.cust_name","total_order_amount").distinct().display()

In [0]:
#Find average order amount by each customer
df_customer.alias("c").join(df_order.alias("o"),col("c.cust_id")==col("o.cust_id"),"inner").groupBy("c.cust_id","c.cust_name").agg(floor(avg(col("o.order_amount"))).alias("avg_order_amount")).orderBy(col("c.cust_id").asc()).select("c.cust_id","c.cust_name","avg_order_amount").display()


In [0]:
#Find highest order amount customer-wise
df_customer.alias("c").join(df_order.alias("o"),col("c.cust_id")==col("o.cust_id"),"inner").groupBy("c.cust_id","c.cust_name").agg(floor(max(col("o.order_amount"))).alias("highest_order_amount")).orderBy(col("c.cust_id").asc()).select("c.cust_id","c.cust_name","highest_order_amount").display()

df_customer.alias("c").join(df_order.alias("o"),col("c.cust_id")==col("o.cust_id"),"inner")\
    .withColumn("total_order_amount",max(col("o.order_amount")).over(Window.partitionBy("c.cust_id"))).select("c.cust_id","c.cust_name","total_order_amount").distinct().display()


In [0]:
%sql
SELECT *
FROM (
    SELECT c.cust_name,
           SUM(o.order_amount) AS total_purchase,
           RANK() OVER(
               ORDER BY SUM(o.order_amount) DESC
           ) AS rnk
    FROM customer c
    JOIN orders o
    ON c.cust_id = o.cust_id
    GROUP BY c.cust_name
) t
WHERE rnk = 1;


In [0]:
#1. Get employee name with department name
df_employee.alias("c").join(df_dept.alias("d"),col("c.dept_id")==col("d.dept_id"),"inner").select("c.emp_name","d.dept_name").show()

In [0]:
#Get all employees including those without departments
df_employee.alias("e").join(df_dept.alias("d"),col("e.dept_id")==col("d.dept_id"),"left").select("e.emp_name","d.dept_name").show()

In [0]:
#Find total salary department-wise
df_dept.alias("d").join(df_employee.alias("e"),col("d.dept_id")==col("e.dept_id"),"inner").groupBy("d.dept_id").agg(sum(col("e.salary").alias("total_salary"))).show()

In [0]:
# Rank employees by salary within department

window_spec=Window.partitionBy("dept_id").orderBy(col("salary").desc())

df_employee_ranked=df_employee.withColumn("rank",dense_rank().over(window_spec)).select("dept_id","rank")

df_employee_ranked.alias("e").join(df_dept.alias("d"),col("e.dept_id")==col("d.dept_id")).select("d.dept_name","e.rank").show()

df_employee.withColumn("rank",dense_rank().over(window_spec)).select("emp_id","dept_id","salary","rank").display()


In [0]:
#Find average salary by department
df_dept.alias("d").join(df_employee.alias("e"),col("d.dept_id")==col("e.dept_id"),"inner").groupBy("d.dept_id").agg(avg(col("e.salary").alias("total_salary"))).show()

In [0]:
#Find highest salary in each department
df_dept.alias("d").join(df_employee.alias("e"),col("d.dept_id")==col("e.dept_id"),"inner").groupBy("d.dept_id").agg(max(col("e.salary"))).show()

In [0]:
# Find highest paid employee in each department
df_employee.withColumn("rank",row_number().over(Window.partitionBy("dept_id").orderBy(col("salary").desc()))).filter(col("rank")==1).select("emp_name","dept_id","salary").show()

In [0]:
#Running total of salaries department-wise
df_employee.alias("e").join(df_dept.alias("d"),col("e.dept_id")==col("d.dept_id"),"inner").withColumn("running_sal",sum(col("salary")).over(
    Window.partitionBy("e.dept_id")
    .orderBy(col("e.salary").desc())
)).select("d.dept_id","d.dept_name","e.salary","running_sal").show()

In [0]:
#Get employee and manager names
df_employee.alias("e").join(df_employee.alias("m"),col("e.manager_id")==col("m.emp_id"),"inner").select("e.emp_name","m.emp_name").show()

In [0]:
#10. Find employees earning more than their managers
df_employee.alias("e").join(df_employee.alias("m"),col("e.manager_id")==col("m.emp_id"),"inner").filter(col("e.salary")>col("m.salary")).select("e.emp_name","e.salary","m.emp_name","m.salary").show()

df_employee.show()

In [0]:
#Find employees with salary above company average
avg_sal=df_employee.select(floor(avg(col("salary")))).collect()[0][0]
display(avg_sal)
df_employee.select("emp_name","salary").filter(col("salary")>avg_sal).show()

In [0]:

df_employee.withColumn(
    "avg_salary",
    avg("salary").over(Window.partitionBy())
).filter(
    col("salary") > col("avg_salary")
).select(
    "emp_name",
    "salary"
).show()

In [0]:
#Find department with highest total salary
df_employee.groupBy("dept_id").agg(sum(col("salary")).alias("total_salary")).orderBy(col("total_salary").desc()).limit(1).show()

In [0]:
#Find department with highest total salary
df_employee.groupBy("dept_id") \
    .agg(sum("salary").alias("total_salary")) \
    .withColumn(
        "rank",
        dense_rank().over(
            Window.orderBy(col("total_salary").desc())
        )
    ) \
    .filter(col("rank") == 1) \
    .show()


In [0]:
#find duplicate salaries
df_employee.groupBy(
    "salary"
) \
.agg(count("*").alias("count")
) \
.filter(
col("count")<=1
)\
.show()





In [0]:
#Find top 3 salaries in each department
df_employee.withColumn(
    "rank",
    dense_rank().over(
        Window.partitionBy("dept_id").orderBy(col("salary").desc())
    )
).filter(
    col("rank")<=3
)\
.select("dept_id","salary").show()


In [0]:
#write a query to filter duplicate records based on time stamp using windows functions
df_employee.show()

df_employee.withColumn(
    "rank",
    dense_rank().over(
        Window.partitionBy("emp_name") \
            .orderBy(col("hire_date").desc()
    )
)
).filter(col("rank")==1).select("emp_name","hire_date").show()


In [0]:
%sql
select emp_id,count(hire_date) as count from employee group by emp_id  having count>1

In [0]:
display(df_employee)

## Aggregation Pattern

In [0]:
# find emp name,max salary in each department
df_employee.groupBy("dept_id").agg(
    max(col("salary")).alias("max_salary")) \
        .withColumnRenamed("dept_id","dept")\
        .join(df_employee.alias("e"),(col("e.dept_id")==col("dept"))&(col("e.salary")==col("max_salary")),"inner") \
        .select("e.emp_name","dept","e.salary")\
            .orderBy(col("dept"))\
                .show()


# using window function
df_employee.withColumn(
    "rank",
    dense_rank().over(
        Window.partitionBy("dept_id").orderBy(col("salary").desc())
    )
).filter(
    col("rank")==1
).select("dept_id","emp_name","salary")\
    .orderBy(col("dept_id"))\
      .show()


    

In [0]:
# Find departments having more than 1 employee
df_employee.groupBy("dept_id").agg(count("*").alias("count")).filter(col("count")>1).show()

#using window function
df_employee.withColumn(
    "rank",
    count("*").over(
        Window.partitionBy("dept_id")
    )
).filter(
    col("rank")>1 \
    ).select("dept_id","rank").limit(1).show()



In [0]:
#Find average salary by department
df_employee.groupBy("dept_id").agg(floor(avg(col("salary"))).alias("avg_salary")).orderBy(col("dept_id").desc()).show() 

#using window function
df_employee.withColumn("avg_salary",avg("salary").over(
    Window.partitionBy("dept_id")
)).select("dept_id","avg_salary").orderBy(col("dept_id").desc()).show()

## 2. Top-N Pattern

In [0]:
# employee with highest salary
df_employee.orderBy(col("salary").desc()).limit(1).show()

In [0]:
#Second highest salary
df_employee.withColumn(
    "rank",
    dense_rank().over(
        Window.orderBy(col("salary").desc())
    )
).filter(
    col("rank")==2
).select("emp_name","dept_id","salary").show()


In [0]:
#Top 3 salaries
df_employee.withColumn(
    "rank",
    dense_rank().over(
        Window.orderBy(col("salary").desc())
    )
).filter(
    col("rank")<=3
).select("emp_name","dept_id","salary").show()

## 3. Department-wise Ranking

In [0]:
#Highest salary employee in each department
df_employee.withColumn(
    "rank",
    row_number().over(
        Window.partitionBy("dept_id").orderBy(col("salary").desc())
    )
).filter(
    col("rank")==1
).select("emp_name","dept_id","salary").show()

## 4. Self Join Pattern

In [0]:
#Employees earning more than their manager
df_employee.alias("e").join(df_employee.alias("m"),col("e.manager_id")==col("m.emp_id"),"inner").filter(col("e.salary")>col("m.salary")).select("*").show()

In [0]:
#Employee-manager pairs
df_employee.alias("e").join(df_employee.alias("m"),col("e.manager_id")==col("m.emp_id"),"inner").select("e.emp_name","m.emp_name").show()
df_employee.show()

## 5. Correlated Subquery Pattern

In [0]:
#Employees earning above department average
df_employee.withColumn("dept_avg",avg(
    "salary").over(Window.partitionBy("dept_id")))\
        .filter(col("salary")>col("dept_avg")).select("emp_name","dept_id","salary").show()


In [0]:
#Employees earning maximum salary in department
df_employee.withColumn("maxsal",max(
    "salary").over(Window.partitionBy("dept_id")))\
        .filter(col("salary")==col("maxsal")).select("emp_name","dept_id","salary").show()

df_employee.withColumn("rank",dense_rank().over(Window.partitionBy("dept_id").orderBy(col("salary").desc()))).filter(col("rank")==1).select("emp_name","dept_id","salary").show()

6. Duplicate Detection Pattern

In [0]:
#Employees having same salary
df_employee.withColumn("cnt",count("*").over(Window.partitionBy("salary"))).filter(col("cnt")>1).select("emp_name","salary").show()



## Relative Ranking Pattern

In [0]:
#name of employee whose sal greater than two other employee from same dept

from pyspark.sql.functions import col, max,count,dense_rank
from pyspark.sql.window import Window
result = (
    df_employee.groupBy("dept_id")
      .agg(
          count("*").alias("emp_cnt"),
          max("salary").alias("max_salary")
      )
       .filter(col("emp_cnt") > 2)
       .withColumnRenamed("dept_id","dept_no")
      .join(
          df_employee.alias("e1"),
          (col("e1.dept_id") == col("dept_no")) &
          (col("e1.salary") == col("max_salary"))
      )
      .select("e1.emp_name", "e1.dept_id", "e1.salary")
)

result.show()

# using window functions
w_dept = Window.partitionBy("dept_id")
w_rank = Window.partitionBy("dept_id").orderBy(col("salary").desc())

result = (df_employee
          .withColumn("emp_cnt", count("*").over(w_dept))
          .withColumn("rnk", dense_rank().over(w_rank))
            .filter((col("emp_cnt") > 2) & (col("rnk") == 1))
           .select("emp_name", "dept_id", "salary"))

result.show()



